# 6단계: "도와주세요" 녹음으로 위급 재현율 확인

`6_own`의 **`응급` 폴더**에 넣은 녹음(5초에 한 번 "도와주세요")을 읽어서, 지금까지 만든 모델 세 개가 **몇 번 알림을 울리는지** 봅니다.
이 녹음은 **학습에 쓰지 않는 시험용**입니다. CPU로 1~2분이면 됩니다.

위에서부터 순서대로 실행하고, **4번, 5번 항목의 출력**을 복사해서 알려 주세요.

## 0. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 파일 찾기

In [ ]:
import glob, os, re, subprocess, pickle, time, csv, unicodedata
import numpy as np
import pandas as pd

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name}을 못 찾았어요. 드라이브에 올렸는지 확인하세요."
    return hits[0]

OWN = find("6_own")
EMB = find("embeddings_v2.npz")
OUT_DIR = os.path.dirname(EMB)
files = sorted(glob.glob(f"{OWN}/*/*.m4a") + glob.glob(f"{OWN}/*/*.wav") + glob.glob(f"{OWN}/*/*.mp3"))
print("녹음 파일", len(files), "개 / 작업 폴더:", OUT_DIR)

## 2. YAMNet 불러오기

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
with tf.io.gfile.GFile(yamnet.class_map_path().numpy()) as f:
    names = [row["display_name"] for row in csv.DictReader(f)]
SPEECH = names.index("Speech")
SCREAM = [names.index(n) for n in ("Screaming", "Shout", "Yell")]
print(len(names), "클래스")

## 3. 새 녹음 변환·임베딩 (캐시된 것은 건너뜀)

In [ ]:
FFMPEG = "ffmpeg"
SR, HOP, WIN = 16000, 7680, 15600
SEG = HOP * 600          # 288초씩 나눠서 처리 (프레임 격자는 이어짐)

def probe_channels(path):
    r = subprocess.run([FFMPEG, "-hide_banner", "-i", path], capture_output=True, text=True, errors="replace")
    m = re.search(r"Audio: .*?, \d+ Hz, ([^,\n]+)", r.stderr)
    return 1 if (m and m.group(1).strip().startswith(("mono", "1 channel"))) else 2

def load_mono(path):
    ch = probe_channels(path)
    raw = subprocess.run([FFMPEG, "-v", "error", "-i", path, "-vn", "-ac", str(ch), "-ar", str(SR), "-f", "s16le", "-"],
                         capture_output=True).stdout
    x = np.frombuffer(raw[: len(raw) // (2 * ch) * (2 * ch)], dtype=np.int16).reshape(-1, ch)
    if ch == 1:
        return x[:, 0].copy(), "모노", None
    a, b = x[::10, 0].astype(np.float64), x[::10, 1].astype(np.float64)
    corr = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else 1.0
    if corr < -0.3:
        return x[:, int(b.std() > a.std())].copy(), "한 채널(위상 반대)", corr
    return x.astype(np.float32).mean(axis=1).astype(np.int16), "두 채널 평균", corr

def embed_stream(x):
    embs, sp, sc, db, ts = [], [], [], [], []
    n = len(x)
    for start in range(0, n, SEG):
        seg = x[start:start + SEG + (WIN - HOP)]
        if len(seg) < WIN:
            break
        w = seg.astype(np.float32) / 32768
        scores, emb, _ = yamnet(tf.constant(w))
        scores, emb = scores.numpy(), emb.numpy()
        last = start + SEG + (WIN - HOP) >= n
        nb = len(w) // HOP
        bp = (w[: nb * HOP].reshape(nb, HOP) ** 2).mean(axis=1)
        fdb = 10 * np.log10(((bp[:-1] + bp[1:]) / 2 if nb > 1 else bp) + 1e-12)
        k = min(len(emb) if last else 600, len(emb), len(fdb))
        embs.append(emb[:k].astype(np.float16))
        sp.append(scores[:k, SPEECH].astype(np.float16))
        sc.append(scores[:k][:, SCREAM].max(axis=1).astype(np.float16))
        db.append(fdb[:k].astype(np.float32))
        ts.append(start / SR + np.arange(k) * 0.48)
    return tuple(np.concatenate(a) for a in (embs, sp, sc, db, ts))


PKL = f"{OUT_DIR}/own_embeddings_v1.pkl"
own = pickle.load(open(PKL, "rb")) if os.path.exists(PKL) else {}
nfc = lambda s: unicodedata.normalize("NFC", s)
for v in own.values():
    v["split"], v["name"] = nfc(v["split"]), nfc(v["name"])
for f in files:
    if f in own:
        continue
    x, how, corr = load_mono(f)
    emb, sp, sc, db, ts = embed_stream(x)
    own[f] = dict(split=nfc(os.path.basename(os.path.dirname(f))), name=nfc(os.path.basename(f)), emb=emb, speech=sp,
                  scream=sc, db=db, t=ts, dur=len(x) / SR, how=how, corr=corr)
    print(f"새로 처리: {own[f]['split']}/{own[f]['name'][:30]} {own[f]['dur']:.0f}초 | {how}", flush=True)
pickle.dump(own, open(PKL, "wb"))
E_files = [f for f, v in own.items() if "응급" in v["split"]]
assert E_files, "응급 폴더의 녹음을 못 찾았어요. 6_own 안에 폴더 이름에 '응급'이 들어가는지 확인하세요."
for f in E_files:
    print(f"응급 소리 녹음: {own[f]['name']} {own[f]['dur']:.0f}초")

## 4. 알림이 울린 발화 수
- 녹음에서 **말소리가 잡힌 덩어리**를 발화 하나로 보고, 그 앞뒤 1초 안에 알림이 울렸으면 "감지"로 셉니다. (쉬는 시간 1초)
- 덩어리가 16개로 나뉘지 않으면 `SPEECH_TH`(말소리 기준)나 `GRID_OFFSET`(첫 발화 시각, 5초 간격 고정)을 바꿔 주세요.
- **발화밖알림**: 발화가 아닌 곳에서 울린 알림 (배경 소음이나 오경보)

In [ ]:
from tensorflow import keras
SPEECH_TH, GAP, EXPECTED = 0.3, 1.5, 16
GRID_OFFSET = None       # 예: 1.2 로 두면 첫 발화 1.2초, 이후 5초 간격 16개로 고정

def alarm_idx(p, t, N, thr, cooldown=1.0):
    ok = (p >= thr).astype(int)
    if len(ok) < N:
        return []
    run = np.convolve(ok, np.ones(N, int), "valid") == N
    out, last = [], -1e9
    for i in np.flatnonzero(run) + (N - 1):
        if t[i] - last >= cooldown:
            out.append(int(i)); last = t[i]
    return out

def find_clusters(v):
    if GRID_OFFSET is not None:
        return [(GRID_OFFSET + 5 * k, GRID_OFFSET + 5 * k + 1.5) for k in range(EXPECTED)]
    sp, t = v["speech"].astype(np.float32), v["t"]
    out = []
    for i in np.flatnonzero(sp > SPEECH_TH):
        if out and t[i] - t[out[-1][1]] <= GAP:
            out[-1][1] = i
        else:
            out.append([i, i])
    return [(float(t[a]), float(t[b]) + 0.96) for a, b in out]

clusters = {f: find_clusters(own[f]) for f in E_files}
n_cl = sum(len(c) for c in clusters.values())
print(f"말소리가 잡힌 발화 덩어리: {n_cl}개 (기대 {EXPECTED}개 x 파일 {len(E_files)}개)")

MODELS = [("03b", "classifier_mlp_v2.keras"), ("04", "classifier_mlp_v3_own.keras"), ("05", "classifier_mlp_v4_own_all.keras")]
probs, rows = {}, []
for label, fn in MODELS:
    try:
        m = keras.models.load_model(find(fn))
    except AssertionError as e:
        print("건너뜀:", e); continue
    pf = {f: m.predict(own[f]["emb"].astype(np.float32), batch_size=4096, verbose=0).ravel() for f in E_files}
    probs[label] = pf
    for N in (1, 2):
        for thr in (0.5, 0.8, 0.9, 0.95):
            hit = tot = outside = 0
            for f in E_files:
                v = own[f]
                times = [v["t"][i - (N - 1)] for i in alarm_idx(pf[f], v["t"], N, thr)]
                inside = lambda x: any(c0 - 1.0 <= x <= c1 + 1.0 for c0, c1 in clusters[f])
                hit += sum(any(c0 - 1.0 <= x <= c1 + 1.0 for x in times) for c0, c1 in clusters[f])
                tot += len(clusters[f]); outside += sum(not inside(x) for x in times)
            rows.append({"모델": label, "N": N, "임계값": thr, "감지한발화": f"{hit}/{tot}", "재현율": hit / max(tot, 1), "발화밖알림": outside})
pd.set_option("display.width", 200)
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 5. 발화마다 결과 (가장 최근 모델 기준)
각 발화의 시각, 그 구간의 **최대 확률**, 설정별 감지(O/X)입니다. 시각 순서가 녹음한 장소 순서(화장실 → ... → 거실)와 같으니, **어느 장소에서 놓치는지** 볼 수 있습니다.

In [ ]:
mmss = lambda s: f"{int(s) // 60}:{int(s) % 60:02d}"
label = [l for l in ("05", "04", "03b") if l in probs][0]
OPS = [(1, 0.5), (1, 0.95), (2, 0.5), (2, 0.9)]
print(f"모델 {label} | 열: " + ", ".join(f"N={n}·{t}" for n, t in OPS))
k = 0
for f in E_files:
    v, p = own[f], probs[label][f]
    hits = {op: [v["t"][i - (op[0] - 1)] for i in alarm_idx(p, v["t"], *op)] for op in OPS}
    for c0, c1 in clusters[f]:
        k += 1
        sel = (v["t"] >= c0 - 1.0) & (v["t"] <= c1 + 1.0)
        mark = " ".join("O" if any(c0 - 1.0 <= x <= c1 + 1.0 for x in hits[op]) else "X" for op in OPS)
        print(f"{k:2d}. {mmss(c0)}  최대확률 {p[sel].max() if sel.any() else 0:.2f}  | {mark}")